# S4.5b — neural-gate tau frontier (Kaggle T4)

**RUNNER ONLY.** Verifier-A decides PASS/FAIL; symbolic rules still drive Reflector diagnostics; Verifier-B is loaded only after the loop and generator are released.

Kaggle: Internet ON, GPU T4, add three inputs: Gemma-3-12b-it weights, `bn_clean.csv`, and the trained Verifier-B directory containing `model.safetensors`, `config.json`, and tokenizer files. Save & Run All is safe; every experiment command is a checked subprocess.


In [ ]:
from pathlib import Path
import os, subprocess
REPO = Path('/kaggle/working/tau_repo')
if (REPO / '.git').is_dir():
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)
else:
    subprocess.run(['git','clone','-q','https://github.com/alphapie77/BSc_Thesis.git',str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git','log','--oneline','-1'], check=True)
for p in ['src/eval/run_tau_traces.py','src/eval/fit_tau.py','configs/s4_tau_traces.yaml','configs/s4_tau.yaml']:
    assert Path(p).is_file(), f'missing from checkout: {p}'


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','uninstall','-y','sklearn-compat'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','scikit-learn==1.9.0','transformers==5.14.1','sentence-transformers==5.6.1','accelerate','bitsandbytes','chromadb','pyyaml','joblib','pytest'], check=True)
gate = "import sklearn,transformers; print(sklearn.__version__,transformers.__version__); assert sklearn.__version__=='1.9.0'"
subprocess.run([sys.executable,'-c',gate], check=True)


In [ ]:
from pathlib import Path
import glob, os, shutil, subprocess, sys
os.chdir('/kaggle/working/tau_repo')
clean = glob.glob('/kaggle/input/**/bn_clean.csv', recursive=True)
assert clean, 'Add the bn-clean Kaggle dataset'
Path('data/cleaned').mkdir(parents=True, exist_ok=True)
shutil.copy2(clean[0], 'data/cleaned/bn_clean.csv')
gemma = [str(Path(p).parent) for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma-3-12b-it' in p.lower()]
vb = [str(Path(p).parent) for p in glob.glob('/kaggle/input/**/model.safetensors', recursive=True) if 'gemma' not in p.lower()]
assert gemma, 'Add google/gemma-3-12b-it as a Kaggle Model input'
assert vb, 'Add the trained Verifier-B directory as a Kaggle input'
MODEL_PATH, VERIFIER_B_PATH = gemma[0], vb[0]
print('Gemma:', MODEL_PATH)
print('Verifier-B:', VERIFIER_B_PATH)
subprocess.run([sys.executable,'src/agents/build_index.py','--config','configs/s4_index.yaml'], check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_s4_critic.py','tests/test_s4_graph.py','tests/test_s4_tau.py','-q'], check=True)


In [ ]:
import os, subprocess, sys
os.chdir('/kaggle/working/tau_repo')
subprocess.run([sys.executable,'src/eval/run_tau_traces.py','--config','configs/s4_tau_traces.yaml','--model-path',MODEL_PATH,'--verifier-b-path',VERIFIER_B_PATH], check=True)
subprocess.run([sys.executable,'src/eval/fit_tau.py','--config','configs/s4_tau.yaml'], check=True)


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
os.chdir('/kaggle/working/tau_repo')
snapshot = 'results/env_snapshot_s4tau_kaggle.json'
subprocess.run([sys.executable,'src/common/env_snapshot.py','--out',snapshot], check=True)
files = ['results/s4_tau_max_traces.jsonl','results/s4_tau_frontier.json',snapshot,'data/generated/s4_tau_calls.jsonl']
for src in map(Path, files):
    assert src.is_file(), f'missing output: {src}'
    dst = Path('/kaggle/working') / src.name
    shutil.copy2(src, dst)
    print('saved', dst, dst.stat().st_size, 'bytes')
